# 🌿 Notebook 2 — Baseline CNN (from scratch)
**Project:** Deforestation Detection from Satellite Imagery  
**Author:** Asliddin | Presidential School, Namangan

---
We train a **simple 4-block CNN from scratch** as our baseline.  
This tells us what's achievable without pre-trained weights — setting the bar for transfer learning in Notebook 3.

**Expected results:** ~83% accuracy, F1 ~0.82

In [ ]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd

from dataset import get_dataloaders
from model import build_model
from evaluate import compute_metrics, plot_confusion_matrix, plot_training_curves

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load Data

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir='../data/processed',
    image_size=224,
    batch_size=32,
)

# Inspect one batch
images, labels = next(iter(train_loader))
print(f'Batch shape: {images.shape}')   # (32, 3, 224, 224)
print(f'Labels: {labels[:10].tolist()}')

## 2. Build Baseline CNN

In [ ]:
model = build_model('baseline').to(device)
print(model)

# Sanity check
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(device)
    out = model(dummy)
    print(f'\nOutput shape: {out.shape}')  # (2, 2)

## 3. Train

In [ ]:
# Training config
EPOCHS     = 15
LR         = 1e-3
SAVE_PATH  = '../models/baseline_cnn.pt'

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'train_f1': [], 'val_f1': []}
best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss, train_preds, train_labels = 0, [], []
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_preds.extend(logits.argmax(1).cpu().tolist())
        train_labels.extend(labels.cpu().tolist())

    # ── Val ──
    model.eval()
    val_loss, val_preds, val_labels = 0, [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            val_loss += criterion(logits, labels).item() * images.size(0)
            val_preds.extend(logits.argmax(1).cpu().tolist())
            val_labels.extend(labels.cpu().tolist())

    scheduler.step()

    t_m = compute_metrics(train_labels, train_preds)
    v_m = compute_metrics(val_labels,   val_preds)
    t_loss = train_loss / len(train_loader.dataset)
    v_loss = val_loss   / len(val_loader.dataset)

    for key, val in [('train_loss', t_loss), ('val_loss', v_loss),
                     ('train_acc', t_m['accuracy']), ('val_acc', v_m['accuracy']),
                     ('train_f1', t_m['f1']), ('val_f1', v_m['f1'])]:
        history[key].append(val)

    print(f'Epoch {epoch:02d}/{EPOCHS} | Train Acc: {t_m["accuracy"]:.4f} F1: {t_m["f1"]:.4f} | Val Acc: {v_m["accuracy"]:.4f} F1: {v_m["f1"]:.4f}')

    if v_m['f1'] > best_f1:
        best_f1 = v_m['f1']
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  ✓ Saved (best Val F1: {best_f1:.4f})')

print(f'\nBaseline training complete. Best Val F1: {best_f1:.4f}')

## 4. Results

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_range = range(1, EPOCHS + 1)

for ax, metric, title in zip(axes, ['loss', 'acc', 'f1'], ['Loss', 'Accuracy', 'F1 Score']):
    ax.plot(epochs_range, history[f'train_{metric}'], label='Train', linewidth=2)
    ax.plot(epochs_range, history[f'val_{metric}'],   label='Val',   linewidth=2, linestyle='--')
    ax.set_title(f'Baseline CNN — {title}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/baseline_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix on val set
plot_confusion_matrix(val_labels, val_preds, save_path='../results/baseline_confusion_matrix.png')

print(f'\nBaseline CNN Final Metrics:')
for k, v in compute_metrics(val_labels, val_preds).items():
    print(f'  {k.capitalize():>10}: {v:.4f}')

print('\n→ Next: Notebook 03 — Transfer learning with ResNet-18')